# 03 — LunarLander Experiment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mattral/rssmlite/blob/main/notebooks/03_lunarlander_experiment.ipynb)

Training `rssmlite` on `LunarLander-v3` — the hardest environment in v1 scope
(SPEC.md Section 7). This env combines shaped dense rewards (hovering near the
pad) with sparse terminal bonuses/penalties on landing/crashing, making the
reward head and long-horizon credit assignment significantly harder than CartPole.

**Expected runtime on a free Colab T4: ~60 minutes** for 500 000 steps.
**Requires box2d:** `pip install swig && pip install gymnasium[box2d]`


In [ ]:
# ── Bootstrap ─────────────────────────────────────────────────────────────
!pip install -q swig
!pip install -q "gymnasium[box2d]"
!pip install -q rssmlite[envs,viz]

from google.colab import drive
drive.mount("/content/drive")

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Device: {gpu}")
assert torch.cuda.is_available(), "Switch to GPU: Runtime → Change runtime type → T4 GPU"


In [ ]:
import os
if not os.path.exists("/content/rssmlite"):
    !git clone -q https://github.com/Mattral/rssmlite.git /content/rssmlite


In [ ]:
import torch, gymnasium as gym, time, re
from rssmlite import RSSMAgent

CHECKPOINT_DIR = "/content/drive/MyDrive/rssmlite-ckpts/lunarlander"
DEVICE = torch.device("cuda")

env = gym.make("LunarLander-v3")
print(f"obs_dim={env.observation_space.shape[0]}, "
      f"action_dim={env.action_space.n}, discrete=True")

agent = RSSMAgent.from_config("/content/rssmlite/configs/lunarlander.yaml", env=env)
agent.rssm.to(DEVICE)
agent.actor.to(DEVICE)
agent.critic.to(DEVICE)
print("Agent ready.")


In [ ]:
# ── Training ───────────────────────────────────────────────────────────────
episode_rewards, env_steps_log = [], []

def log_fn(msg):
    m = re.match(r"\[step (\d+)\] episode_reward=([\d.\-]+)", msg)
    if m:
        env_steps_log.append(int(m.group(1)))
        episode_rewards.append(float(m.group(2)))
    if len(episode_rewards) % 50 == 0:
        print(msg)

t0 = time.time()
agent.train(
    env,
    steps=500_000,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_every=25_000,
    log_every=1,
    log_fn=log_fn,
)
elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed/60:.1f} min")
env.close()


In [ ]:
# ── Training curve ─────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

def smooth(x, w=50):
    return np.convolve(x, np.ones(w)/w, mode="valid") if len(x) >= w else x

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(env_steps_log, episode_rewards, alpha=0.2, color="steelblue", label="raw")
if len(episode_rewards) >= 50:
    ax.plot(env_steps_log[49:], smooth(episode_rewards), color="steelblue",
            linewidth=2, label="smoothed (w=50)")
ax.axhline(200, color="tomato", linestyle="--", label="LunarLander solve threshold (200)")
ax.set_xlabel("environment steps")
ax.set_ylabel("episode return")
ax.set_title("rssmlite on LunarLander-v3")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("lunarlander_training_curve.png", dpi=150)
plt.show()


In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
from rssmlite import run_evaluation_episodes

eval_env = gym.make("LunarLander-v3")
results = run_evaluation_episodes(agent, eval_env, n_episodes=20)
eval_env.close()

print(f"Evaluation over 20 episodes:")
print(f"  Mean return : {results['mean_return']:.1f}")
print(f"  Std return  : {results['std_return']:.1f}")
print(f"  Min / Max   : {results['min_return']:.1f} / {results['max_return']:.1f}")
print(f"  Solved (>=200): {results['mean_return'] >= 200}")


In [ ]:
# ── Reconstruction quality ─────────────────────────────────────────────────
from rssmlite import reconstruction_report
from rssmlite.evaluation import plot_reconstruction

batch = agent.buffer.sample(batch_size=32, seq_len=50)
batch_dev = {k: v.to(DEVICE) for k, v in batch.items()}
report = reconstruction_report(agent, batch_dev)

print("Reconstruction MSE (symlog space):", f"{report['mse_symlog']:.4f}")
print("Reconstruction MSE (real units)  :", f"{report['mse_real']:.4f}")
labels = ["x", "y", "vx", "vy", "angle", "angular_vel", "left_leg", "right_leg"]
for i, (lbl, v) in enumerate(zip(labels, report["per_dim_mse"].cpu().tolist())):
    print(f"  {lbl:15s}: {v:.4f}")


In [ ]:
# ── Imagined rollout ───────────────────────────────────────────────────────
rollout = agent.imagine_rollout(steps=20)
rewards = rollout["reward_pred"].squeeze().cpu().detach().numpy()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(rewards, marker="o", color="steelblue")
ax.set_xlabel("imagined timestep")
ax.set_ylabel("predicted reward")
ax.set_title("Imagined rollout: predicted reward sequence")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("lunarlander_imagined_rewards.png", dpi=150)
plt.show()


In [ ]:
# ── Latent space ──────────────────────────────────────────────────────────
fig = agent.visualize_latent_space()
plt.savefig("lunarlander_latent_space.png", dpi=150)
plt.show()


## Notes on LunarLander difficulty

LunarLander is harder than CartPole for the world model in two ways:

1. **Longer horizon:** the reward for a successful landing arrives hundreds
   of steps after takeoff. The critic's lambda-return must propagate this
   signal backward through the imagined rollout, which requires a good
   long-range dynamics model.

2. **Mixed reward structure:** a shaped dense reward (hovering near the pad)
   combined with a sparse terminal bonus/penalty means the reward head has to
   learn two qualitatively different signal types simultaneously.

If training hasn't converged after 500 000 steps, try:
- Increasing `steps` to 1 000 000 in `configs/lunarlander.yaml`.
- Increasing `seed_episodes` to 50 (more diverse initial data).
- Increasing `replay_capacity_episodes` to 2000.

See `ROADMAP.md P2` for the planned `TransformerDynamics` ablation, which
may improve long-range credit assignment on this environment.
